# Credit Card Fraud Detection & Risk Analysis
Every number is computed when you run the cell. Set `USE_SYNTHETIC=True` only to test the code (results are then NOT real).

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, joblib
import fraud_pipeline as fp
USE_SYNTHETIC = False      # True = smoke test on fake data
DATA = 'data/creditcard.csv'
FAST = False
OUT = Path('outputs'); OUT.mkdir(exist_ok=True)

## 1-2. Load, inspect, clean
Target `Class`: 1 = fraud. `V1..V28` are anonymised PCA components; `Time` = seconds since first transaction; `Amount` = transaction amount.

In [ ]:
df = fp.make_synthetic_data() if USE_SYNTHETIC else fp.load_data(DATA)
info = fp.inspect_data(df); info

In [ ]:
df = fp.engineer_features(fp.clean_data(df))
print(df.shape); df[['Amount','log_amount','hour_of_day','is_night']].describe()

## 3. EDA
Each saved plot is shown below; interpret it using the measured insights printed.

In [ ]:
eda = fp.run_eda(df, OUT)
from IPython.display import Image, display
for f in ['eda_class_balance','eda_amount','eda_time','eda_corr_heatmap','eda_feature_dists']:
    display(Image(OUT/f'{f}.png'))
eda['top_corr_with_target'], eda['fraud_share_above_legit_p99_amount']

In [ ]:
pd.read_csv(OUT/'eda_stat_tests.csv').head(10)

## 4. Split and class-imbalance strategies
Resampling touches the training data only.

In [ ]:
s = fp.split_data(df)
print(len(s.X_train), len(s.X_val), len(s.X_test), s.y_train.sum(), s.y_val.sum(), s.y_test.sum())
imb = fp.compare_imbalance_strategies(s, OUT, FAST); imb

## 5-6. Models, tuning, evaluation
Tuning on train (CV, PR-AUC); threshold chosen on validation; test set is held out.

In [ ]:
comp, fitted, vsc, tsc = fp.train_and_evaluate(s, OUT, FAST)
fp.plot_model_curves(s, tsc, comp, OUT)
comp[comp.split=='validation'].round(4)

In [ ]:
comp[comp.split=='test'].round(4)

In [ ]:
for f in ['model_roc_pr_curves','model_metric_bars','model_confusion_matrices']:
    display(Image(OUT/f'{f}.png'))

In [ ]:
final = fp.select_final_model(comp)
thr = float(comp[(comp.model==final)&(comp.split=='validation')].threshold.iloc[0])
final, thr

## 7. False positives / false negatives

In [ ]:
raw_test = s.X_test.copy(); raw_test['Amount'] = np.expm1(raw_test['log_amount'])
err = fp.error_analysis(s, final, tsc[final], thr, raw_test, OUT)
display(Image(OUT/'error_analysis.png')); err

In [ ]:
pd.read_csv(OUT/'threshold_tradeoff_test.csv')

## 8. Feature importance (association, not causation)

In [ ]:
imp = fp.feature_importance(fitted[final], s, OUT)
display(Image(OUT/'feature_importance_permutation.png')); imp['permutation_top10']

## 9. Risk scoring
Scores are ranking scores, not calibrated probabilities. 'Flagged' means review/step-up, not confirmed fraud.

In [ ]:
cfg = fp.RiskConfig(medium=0.30, high=0.70)
fp.risk_distribution(s, tsc[final], cfg, OUT)

In [ ]:
sample = s.X_test.head(5)
fp.score_transactions(fitted[final], sample, cfg)

## 10-11. Save artifacts, dashboard and report
Run `python run_pipeline.py` to generate `outputs/final_report.md` and then `streamlit run streamlit_app.py`.

In [ ]:
joblib.dump({'model': fitted[final], 'name': final, 'threshold': thr, 'features': s.feature_names}, OUT/'final_model.joblib')